In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 77 (delta 21), reused 17 (delta 17), pack-reused 48 (from 1)
Receiving objects: 100% (77/77), 1.05 MiB | 1.78 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [3]:
pip install ucimlrepo

In [5]:
pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.9/206.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 9.6 MB/s eta 0:00:00


In [6]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------

magic_gamma_telescope = fetch_ucirepo(id=159)

# data (as pandas dataframes)
X = magic_gamma_telescope.data.features
y = magic_gamma_telescope.data.targets

# Combine features and target into a single DataFrame
data = pd.concat([X, y], axis=1)

target_col = y.columns[0] # Correctly identify the target column

# metadata
print("Dataset Metadata:")
print(magic_gamma_telescope.metadata)

# variable information
print("\nDataset Variable Information:")
print(magic_gamma_telescope.variables)

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data) # Use the combined DataFrame

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []

Dataset Metadata:
{'uci_id': 159, 'name': 'MAGIC Gamma Telescope', 'repository_url': 'https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope', 'data_url': 'https://archive.ics.uci.edu/static/public/159/data.csv', 'abstract': 'Data are MC generated to simulate registration of high energy gamma particles in an atmospheric Cherenkov telescope', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 19020, 'num_features': 10, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2004, 'last_updated': 'Tue Dec 19 2023', 'dataset_doi': '10.24432/C52C8B', 'creators': ['R. Bock'], 'intro_paper': None, 'additional_info': {'summary': "The data are MC generated (see below) to simulate registration of high energy gamma particles in a ground-based atmospheric Cherenkov gamma telescope using the imaging techniq

In [7]:
# SINGLE RUN

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


# TRAIN / TEST SPLIT (NO LEAKAGE)

train_real, test_real = train_test_split(
    data,
    test_size=TEST_SIZE,
    stratify=data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# CTABGAN
# ---------------------------------------------------

try:

    data_path = "magic_gamma_telescope_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[target_col],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Classification": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)


================ SINGLE RUN ================


100%|██████████| 150/150 [25:33<00:00, 10.22s/it]


Finished training in 1555.9371781349182  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 355.52it/s]|
Column Shapes Score: 96.05%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 130.99it/s]|
Column Pair Trends Score: 93.03%

Overall Score (Average): 94.54%

CTABGAN: 0.9454


In [8]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()
    data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, 1)
        .astype(int)
    )

    synthetic_wgan[target_col] = encoder.inverse_transform(
        synthetic_wgan[target_col]
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 316.12it/s]|
Column Shapes Score: 92.76%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 125.97it/s]|
Column Pair Trends Score: 95.01%

Overall Score (Average): 93.88%

WGAN_GP: 0.9388


In [9]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 313.71it/s]|
Column Shapes Score: 89.8%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 94.98it/s]|
Column Pair Trends Score: 90.28%

Overall Score (Average): 90.04%

CTGAN: 0.9004
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 326.00it/s]|
Column Shapes Score: 89.34%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 138.95it/s]|
Column Pair Trends Score: 89.33%

Overall Score (Average): 89.34%

CopulaGAN: 0.8934
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 282.09it/s]|
Column Shapes Score: 88.77%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 136.65it/s]|
Column Pair Trends Score: 92.67%

Overall Score (Average): 90.72%

TVAE: 0.9072
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 349.17it/s]|
Column Shapes Sc

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51]
):

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=y_train
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=y_test
            )

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} \u00b1 {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} \u00b1 {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} \u00b1 {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )

In [21]:
import pandas as pd

label_col = target_col

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=data,
    test_df=data,
    label="class",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=data,
        label="class",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=["_TRTR", "_TSTR"]
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
5,RandomForest,0.8803 ± 0.0056,0.9106 ± 0.0042,0.8825 ± 0.0041,0.9407 ± 0.0051
6,ExtraTrees,0.8783 ± 0.0062,0.9100 ± 0.0046,0.8740 ± 0.0054,0.9492 ± 0.0053
9,MLP,0.8780 ± 0.0050,0.9089 ± 0.0038,0.8810 ± 0.0056,0.9386 ± 0.0075
7,GradientBoost,0.8715 ± 0.0048,0.9054 ± 0.0034,0.8657 ± 0.0043,0.9490 ± 0.0035
1,SVM-RBF,0.8694 ± 0.0051,0.9047 ± 0.0035,0.8583 ± 0.0051,0.9565 ± 0.0034
2,KNN,0.8379 ± 0.0045,0.8826 ± 0.0032,0.8318 ± 0.0037,0.9401 ± 0.0043
8,AdaBoost,0.8261 ± 0.0073,0.8699 ± 0.0062,0.8442 ± 0.0063,0.8974 ± 0.0139
4,DecisionTree,0.8164 ± 0.0046,0.8581 ± 0.0034,0.8599 ± 0.0051,0.8564 ± 0.0040
0,LogReg,0.7909 ± 0.0075,0.8477 ± 0.0054,0.8032 ± 0.0059,0.8974 ± 0.0058
3,NaiveBayes,0.7257 ± 0.0080,0.8125 ± 0.0053,0.7297 ± 0.0055,0.9165 ± 0.0061


CTGAN - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
1,SVM-RBF,0.7558 ± 0.0085,0.8079 ± 0.0077,0.8241 ± 0.0050,0.7923 ± 0.0121
0,LogReg,0.7497 ± 0.0078,0.8097 ± 0.0067,0.7983 ± 0.0048,0.8214 ± 0.0111
3,NaiveBayes,0.7475 ± 0.0087,0.8215 ± 0.0055,0.7583 ± 0.0087,0.8965 ± 0.0086
6,ExtraTrees,0.7466 ± 0.0082,0.7975 ± 0.0076,0.8272 ± 0.0055,0.7700 ± 0.0124
9,MLP,0.7351 ± 0.0128,0.7915 ± 0.0112,0.8079 ± 0.0111,0.7761 ± 0.0185
5,RandomForest,0.7317 ± 0.0119,0.7848 ± 0.0110,0.8173 ± 0.0068,0.7548 ± 0.0166
7,GradientBoost,0.7284 ± 0.0103,0.7815 ± 0.0099,0.8166 ± 0.0082,0.7495 ± 0.0172
8,AdaBoost,0.7273 ± 0.0074,0.7806 ± 0.0067,0.8159 ± 0.0103,0.7485 ± 0.0138
2,KNN,0.7134 ± 0.0092,0.7719 ± 0.0098,0.7971 ± 0.0028,0.7485 ± 0.0180
4,DecisionTree,0.6712 ± 0.0102,0.7349 ± 0.0110,0.7698 ± 0.0093,0.7035 ± 0.0210


CTGAN - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
5,RandomForest,0.8210 ± 0.0048,0.8719 ± 0.0032,0.8135 ± 0.0067,0.9395 ± 0.0078
7,GradientBoost,0.8168 ± 0.0075,0.8678 ± 0.0049,0.8155 ± 0.0087,0.9275 ± 0.0073
6,ExtraTrees,0.8106 ± 0.0071,0.8656 ± 0.0043,0.8020 ± 0.0083,0.9401 ± 0.0050
1,SVM-RBF,0.8023 ± 0.0081,0.8620 ± 0.0053,0.7875 ± 0.0074,0.9521 ± 0.0063
8,AdaBoost,0.8012 ± 0.0111,0.8536 ± 0.0095,0.8168 ± 0.0116,0.8944 ± 0.0226
9,MLP,0.7944 ± 0.0088,0.8456 ± 0.0065,0.8241 ± 0.0091,0.8683 ± 0.0103
0,LogReg,0.7834 ± 0.0070,0.8444 ± 0.0048,0.7905 ± 0.0067,0.9062 ± 0.0069
2,KNN,0.7673 ± 0.0088,0.8357 ± 0.0057,0.7706 ± 0.0079,0.9129 ± 0.0060
4,DecisionTree,0.7580 ± 0.0098,0.8183 ± 0.0082,0.7972 ± 0.0087,0.8408 ± 0.0162
3,NaiveBayes,0.7478 ± 0.0064,0.8284 ± 0.0033,0.7410 ± 0.0072,0.9394 ± 0.0062


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,CTABGAN,RandomForest,0.059332,0.038771,0.069037,0.001217,0.8803 ± 0.0056,0.8210 ± 0.0048
1,CTABGAN,ExtraTrees,0.067692,0.044472,0.071987,0.009043,0.8783 ± 0.0062,0.8106 ± 0.0071
2,CTABGAN,MLP,0.083596,0.063306,0.056906,0.070316,0.8780 ± 0.0050,0.7944 ± 0.0088
3,CTABGAN,GradientBoost,0.054706,0.037643,0.050251,0.021573,0.8715 ± 0.0048,0.8168 ± 0.0075
4,CTABGAN,SVM-RBF,0.067061,0.042740,0.070771,0.004420,0.8694 ± 0.0051,0.8023 ± 0.0081
5,CTABGAN,KNN,0.070610,0.046917,0.061152,0.027251,0.8379 ± 0.0045,0.7673 ± 0.0088
6,CTABGAN,AdaBoost,0.024842,0.016329,0.027427,0.003001,0.8261 ± 0.0073,0.8012 ± 0.0111
7,CTABGAN,DecisionTree,0.058386,0.039820,0.062668,0.015612,0.8164 ± 0.0046,0.7580 ± 0.0098
8,CTABGAN,LogReg,0.007466,0.003303,0.012663,-0.008759,0.7909 ± 0.0075,0.7834 ± 0.0070
9,CTABGAN,NaiveBayes,-0.022029,-0.015971,-0.011357,-0.022871,0.7257 ± 0.0080,0.7478 ± 0.0064


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
5,WGAN_GP,0.035999,0.029264,0.016812,0.043167
0,CTABGAN,0.047166,0.031733,0.047150,0.012080
4,TVAE,0.082137,0.078088,0.006352,0.149672
2,CopulaGAN,0.098304,0.077580,0.059033,0.098394
3,GaussianCopula,0.099582,0.063688,0.099099,0.015588
1,CTGAN,0.106777,0.092871,0.039760,0.148082


In [23]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
